In [ ]:
import os, pickle, json, warnings, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL_MAP   = {'Low': 0, 'Medium': 1, 'High': 2}
INV_LABEL   = {0: 'Low', 1: 'Medium', 2: 'High'}
WINDOW_SIZE = 30
N_FEATURES  = 8
OUT_DIR     = '/kaggle/working'
print(f"Device: {DEVICE}")

In [ ]:
class ACLRiskLSTM(nn.Module):
    def __init__(self, n_features=8, hidden_size=128, num_layers=2,
                 num_classes=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout, bidirectional=True
        )
        self.attn = nn.Linear(hidden_size * 2, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, num_classes)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        w   = torch.softmax(self.attn(out), dim=1)
        ctx = (w * out).sum(dim=1)
        return self.classifier(ctx)

In [ ]:
dummy = torch.zeros(4, 30, 8).to(DEVICE)
tformer_test = ACLRiskTransformer().to(DEVICE)
out = tformer_test(dummy)
print("Transformer output shape:", out.shape)   # Should be (4, 3)
assert out.shape == (4, 3)
print("Architecture OK")
del tformer_test

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding.
    Uses register_buffer so it moves to GPU automatically with .to(DEVICE).
    """
    def __init__(self, d_model, max_len=30, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)           # (30, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)                          # (1, 30, d_model)

        # register_buffer: not a parameter, but moves with .to(DEVICE)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (B, T, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class ACLRiskTransformer(nn.Module):
    """
    Temporal Transformer for ACL risk classification.
    Input  : (B, 30, 8)
    Output : (B, 3)
    """
    def __init__(self, n_features=8, d_model=64, nhead=4,
                 num_layers=3, dim_feedforward=256,
                 num_classes=3, dropout=0.1):
        super().__init__()

        # Project raw 8 features → d_model (64)
        self.input_proj = nn.Linear(n_features, d_model)

        self.pos_enc = PositionalEncoding(d_model, max_len=30, dropout=dropout)

        # batch_first=True — CRITICAL: input is (B, T, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )

        # Two-layer classifier head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x: (B, 30, 8)
        x = self.input_proj(x)                      # (B, 30, 64)
        x = self.pos_enc(x)                         # (B, 30, 64)
        x = self.transformer_encoder(x)             # (B, 30, 64)
        x = x.mean(dim=1)                           # (B, 64) — global avg pool
        return self.classifier(x)                   # (B, 3)

In [ ]:
TRAIN_PKL = '/kaggle/input/datasets/haruxo/p5-opt/train_samples.pkl'
VAL_PKL   = '/kaggle/input/datasets/haruxo/p5-opt/val_samples.pkl'

with open(TRAIN_PKL, 'rb') as f:
    train_samples = pickle.load(f)
with open(VAL_PKL, 'rb') as f:
    val_samples = pickle.load(f)

# --- Rebalance training set (same as Phase 6) ---
import random
random.seed(SEED)

low_s  = [s for s in train_samples if s['label'] == 'Low']
med_s  = [s for s in train_samples if s['label'] == 'Medium']
high_s = [s for s in train_samples if s['label'] == 'High']

n_high  = len(high_s)
cap_low = max(4 * n_high, 5000)
cap_med = max(2 * n_high, len(med_s))

low_s = random.sample(low_s, min(cap_low, len(low_s)))
med_s = random.sample(med_s, min(cap_med, len(med_s)))

train_samples = low_s + med_s + high_s
random.shuffle(train_samples)

print(f"Train: {len(train_samples)}  |  Val: {len(val_samples)}")

# --- Dataset ---
class ACLDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        x = torch.tensor(s['features'], dtype=torch.float32)
        y = LABEL_MAP[s['label']]
        return x, y

# --- Class weights ---
from collections import Counter
dist  = Counter(s['label'] for s in train_samples)
total = sum(dist.values())
inv_f = {k: total / (3.0 * v) for k, v in dist.items()}
ns    = sum(inv_f.values())
cw    = {k: v / ns for k, v in inv_f.items()}

class_weights_tensor = torch.tensor(
    [cw['Low'], cw['Medium'], cw['High']], dtype=torch.float32
).to(DEVICE)

# --- Weighted sampler ---
sample_weights = [
    1.0 / dist[s['label']] for s in train_samples
]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_samples), replacement=True)

train_dataset = ACLDataset(train_samples)
val_dataset   = ACLDataset(val_samples)

train_loader = DataLoader(train_dataset, batch_size=256, sampler=sampler,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False,    num_workers=2)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

def quick_eval(mdl, loader):
    """
    Returns macro F1, macro AUROC, High Risk FNR, all predictions/probs.
    FNR computed directly from confusion matrix.
    """
    mdl.eval()
    all_true, all_pred, all_probs = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            logits = mdl(x)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = probs.argmax(axis=1)
            all_true.append(y.numpy())
            all_pred.append(preds)
            all_probs.append(probs)

    all_true  = np.concatenate(all_true)
    all_pred  = np.concatenate(all_pred)
    all_probs = np.concatenate(all_probs)

    f1 = f1_score(all_true, all_pred, average='macro', zero_division=0)

    bin_true = label_binarize(all_true, classes=[0, 1, 2])
    try:
        auroc = roc_auc_score(bin_true, all_probs, average='macro', multi_class='ovr')
    except Exception:
        auroc = 0.0

    # FNR for High Risk (class 2) from confusion matrix
    cm        = confusion_matrix(all_true, all_pred, labels=[0, 1, 2])
    tp_high   = cm[2, 2]
    fn_high   = cm[2, 0] + cm[2, 1]
    fnr_high  = fn_high / (tp_high + fn_high) if (tp_high + fn_high) > 0 else 0.0

    return float(f1), float(auroc), float(fnr_high), all_true, all_pred, all_probs

In [ ]:
EPOCHS   = 40
PATIENCE = 8
LR       = 3e-4

transformer = ACLRiskTransformer().to(DEVICE)
optimizer   = torch.optim.AdamW(transformer.parameters(), lr=LR, weight_decay=1e-4)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_f1_t    = 0.0
patience_counter = 0
train_losses     = []
val_f1s          = []

print("Training Temporal Transformer...")
print(f"Epochs: {EPOCHS}  |  Patience: {PATIENCE}  |  LR: {LR}\n")

for epoch in range(1, EPOCHS + 1):
    transformer.train()
    epoch_loss = 0.0
    n_batches  = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(transformer(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(transformer.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches  += 1

    scheduler.step()
    avg_loss = epoch_loss / n_batches

    val_f1, val_auroc, val_fnr, _, _, _ = quick_eval(transformer, val_loader)
    train_losses.append(avg_loss)
    val_f1s.append(val_f1)

    print(f"Epoch {epoch:02d}/{EPOCHS}  |  Loss: {avg_loss:.4f}  "
          f"|  Val F1: {val_f1:.4f}  |  AUROC: {val_auroc:.4f}  |  FNR: {val_fnr:.4f}")

    if val_f1 > best_val_f1_t:
        best_val_f1_t = val_f1
        patience_counter = 0
        torch.save({
            'epoch'            : epoch,
            'model_state_dict' : transformer.state_dict(),
            'optimizer_state'  : optimizer.state_dict(),
            'best_val_f1'      : best_val_f1_t,
        }, os.path.join(OUT_DIR, 'best_acl_transformer.pt'))
        print(f"  ✓ New best transformer saved  (F1={best_val_f1_t:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}. Best Val F1: {best_val_f1_t:.4f}")
            break

print(f"\nTransformer training complete. Best Val F1: {best_val_f1_t:.4f}")

In [ ]:
LSTM_CKPT        = '/kaggle/input/datasets/haruxo/model-lstm/acl_risk_model_phase6.pt'
TRANSFORMER_CKPT = os.path.join(OUT_DIR, 'best_acl_transformer.pt')

# Load BiLSTM
lstm_model = ACLRiskLSTM().to(DEVICE)
lstm_ckpt  = torch.load(LSTM_CKPT, map_location=DEVICE)
lstm_model.load_state_dict(lstm_ckpt['model_state_dict'])
lstm_model.eval()
print(f"BiLSTM loaded     — Best F1: {lstm_ckpt.get('best_val_f1', 'N/A')}")

# Load Transformer
trans_model = ACLRiskTransformer().to(DEVICE)
trans_ckpt  = torch.load(TRANSFORMER_CKPT, map_location=DEVICE)
trans_model.load_state_dict(trans_ckpt['model_state_dict'])
trans_model.eval()
print(f"Transformer loaded — Best F1: {trans_ckpt.get('best_val_f1', 'N/A')}")

In [ ]:
def ensemble_predict(lstm_mdl, trans_mdl, loader):
    """
    Soft voting: average softmax probabilities from both models.
    Returns true labels, ensemble predictions, ensemble probabilities.
    """
    lstm_mdl.eval()
    trans_mdl.eval()

    all_true, all_ensemble_probs = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)

            lstm_probs  = torch.softmax(lstm_mdl(x),  dim=1).cpu().numpy()
            trans_probs = torch.softmax(trans_mdl(x), dim=1).cpu().numpy()

            # Soft vote: equal weight average
            ens_probs = (lstm_probs + trans_probs) / 2.0

            all_true.append(y.numpy())
            all_ensemble_probs.append(ens_probs)

    all_true          = np.concatenate(all_true)
    all_ensemble_probs = np.concatenate(all_ensemble_probs)
    all_ensemble_pred  = all_ensemble_probs.argmax(axis=1)

    return all_true, all_ensemble_pred, all_ensemble_probs


ens_true, ens_pred, ens_probs = ensemble_predict(lstm_model, trans_model, val_loader)

# Ensemble metrics
ens_f1   = f1_score(ens_true, ens_pred, average='macro', zero_division=0)
bin_true  = label_binarize(ens_true, classes=[0, 1, 2])
try:
    ens_auroc = roc_auc_score(bin_true, ens_probs, average='macro', multi_class='ovr')
except Exception:
    ens_auroc = 0.0

cm_ens   = confusion_matrix(ens_true, ens_pred, labels=[0, 1, 2])
tp_h     = cm_ens[2, 2]
fn_h     = cm_ens[2, 0] + cm_ens[2, 1]
ens_fnr  = fn_h / (tp_h + fn_h) if (tp_h + fn_h) > 0 else 0.0

print(f"Ensemble  —  F1: {ens_f1:.4f}  |  AUROC: {ens_auroc:.4f}  |  FNR: {ens_fnr:.4f}")

In [ ]:
lstm_f1, lstm_auroc, lstm_fnr, _, _, _ = quick_eval(lstm_model,  val_loader)
tran_f1, tran_auroc, tran_fnr, _, _, _ = quick_eval(trans_model, val_loader)

print(f"BiLSTM      —  F1: {lstm_f1:.4f}  |  AUROC: {lstm_auroc:.4f}  |  FNR: {lstm_fnr:.4f}")
print(f"Transformer —  F1: {tran_f1:.4f}  |  AUROC: {tran_auroc:.4f}  |  FNR: {tran_fnr:.4f}")

In [ ]:
comparison_data = {
    'Model'        : ['BiLSTM + Attention', 'Temporal Transformer', 'Ensemble (Soft Vote)'],
    'Val_F1_Macro' : [round(lstm_f1,  4), round(tran_f1,  4), round(ens_f1,  4)],
    'Val_AUROC'    : [round(lstm_auroc,4), round(tran_auroc,4), round(ens_auroc,4)],
    'HighRisk_FNR' : [round(lstm_fnr, 4), round(tran_fnr, 4), round(ens_fnr, 4)],
}

comp_df = pd.DataFrame(comparison_data)

print("\n" + "=" * 62)
print("MODEL COMPARISON TABLE")
print("=" * 62)
print(comp_df.to_string(index=False))
print("=" * 62)

# Highlight winner per metric
best_f1    = comp_df.loc[comp_df['Val_F1_Macro'].idxmax(),  'Model']
best_auroc = comp_df.loc[comp_df['Val_AUROC'].idxmax(),     'Model']
best_fnr   = comp_df.loc[comp_df['HighRisk_FNR'].idxmin(),  'Model']
print(f"\n  Best F1    : {best_f1}")
print(f"  Best AUROC : {best_auroc}")
print(f"  Lowest FNR : {best_fnr}")

comp_df.to_csv(os.path.join(OUT_DIR, 'model_comparison.csv'), index=False)
print("\nSaved: model_comparison.csv")

In [ ]:
torch.save({
    # Both model state dicts
    'lstm_state_dict'        : lstm_model.state_dict(),
    'transformer_state_dict' : trans_model.state_dict(),

    # Architecture configs
    'lstm_config' : {
        'n_features': 8, 'hidden_size': 128,
        'num_layers': 2, 'num_classes': 3, 'dropout': 0.3
    },
    'transformer_config' : {
        'n_features': 8, 'd_model': 64, 'nhead': 4,
        'num_layers': 3, 'dim_feedforward': 256,
        'num_classes': 3, 'dropout': 0.1
    },

    # Metrics
    'lstm_val_f1'        : lstm_f1,
    'transformer_val_f1' : tran_f1,
    'ensemble_val_f1'    : ens_f1,
    'ensemble_val_auroc' : ens_auroc,
    'ensemble_fnr_high'  : ens_fnr,

    # Metadata
    'label_map'    : LABEL_MAP,
    'window_size'  : WINDOW_SIZE,
    'n_features'   : N_FEATURES,
    'ensemble_mode': 'soft_vote_equal_weight',
    'feature_cols' : [
        'l_knee_angle', 'r_knee_angle', 'l_hip_angle', 'r_hip_angle',
        'trunk_lean', 'asymmetry', 'l_knee_vel', 'r_knee_vel'
    ],
    'known_limitations': [
        '2D angle estimates carry up to 18 degree error vs 3D clinical systems',
        'No confirmed injury outcome data in training set',
        'Camera must be front-facing or sagittal within ±10 degrees'
    ]
}, os.path.join(OUT_DIR, 'ensemble_pipeline_phase7.pt'))

print("Saved: ensemble_pipeline_phase7.pt")

In [ ]:
outputs_6g = [
    'best_acl_transformer.pt',
    'model_comparison.csv',
    'ensemble_pipeline_phase7.pt'
]

print("=" * 50)
print("PHASE 6G OUTPUT FILES")
print("=" * 50)
for fname in outputs_6g:
    fpath = os.path.join(OUT_DIR, fname)
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {fname:<38}  {size_kb:>8.1f} KB")
    else:
        print(f"  {fname:<38}  NOT FOUND")

print(f"\nBiLSTM F1       : {lstm_f1:.4f}")
print(f"Transformer F1  : {tran_f1:.4f}")
print(f"Ensemble F1     : {ens_f1:.4f}")
print(f"Ensemble AUROC  : {ens_auroc:.4f}")
print(f"Ensemble FNR    : {ens_fnr:.4f}")
print("\nPhase 6G complete.")